In [15]:
import os
import glob
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
import numpy as np
import faiss

import faiss
import pandas as pd
from sklearn.preprocessing import normalize

In [22]:
files = glob.glob("../data/knowledge_base/*.txt")

for file in files:
    print(os.path.basename(file))

billing_policy.txt
cancellation_policy.txt
customer_service_guidelines.txt
delivery_policy.txt
escalation_policy.txt
payment_policy.txt
refund_policy.txt
technical_support.txt
warranty_policy.txt


In [23]:
# create chunks

chunks = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        text = f.read().strip()

    paragraphs = [paragraph.strip() for paragraph in text.split("\n\n")if paragraph.strip()]

    for chunk_id, paragraph in enumerate(paragraphs):
        chunks.append({
            "document_id": os.path.splitext(os.path.basename(file))[0],
            "source": os.path.basename(file),
            "chunk_id": chunk_id,
            "text": paragraph
        })

knowledge_base_chunks = pd.DataFrame(chunks)

print("Total chunks:", len(knowledge_base_chunks))
display(knowledge_base_chunks.head(10))

Total chunks: 137


,document_id,source,chunk_id,text
0,billing_policy,billing_policy.txt,0,Billing Policy
1,billing_policy,billing_policy.txt,1,Customers are responsible for reviewing their ...
2,billing_policy,billing_policy.txt,2,"If a customer notices an incorrect charge, une..."
3,billing_policy,billing_policy.txt,3,Customers should provide the relevant customer...
4,billing_policy,billing_policy.txt,4,If a payment was successful but the order was ...
5,billing_policy,billing_policy.txt,5,Customers should not make a second payment unt...
6,billing_policy,billing_policy.txt,6,If a customer is charged twice for the same pu...
7,billing_policy,billing_policy.txt,7,If the billed amount differs from the expected...
8,billing_policy,billing_policy.txt,8,Billing records should be checked before confi...
9,billing_policy,billing_policy.txt,9,The assistant must not invent transaction amou...


In [24]:
knowledge_base_chunks.to_csv("../data/processed/knowledge_base_chunks.csv", index=False)

print("Knowledge base chunks saved")
print(knowledge_base_chunks.shape)

Knowledge base chunks saved
(137, 4)


In [6]:
# create knowledge bae embeddings

In [25]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

knowledge_texts = knowledge_base_chunks["text"].tolist()

knowledge_embeddings = embedding_model.encode(
    knowledge_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

knowledge_embeddings = normalize(
    knowledge_embeddings,
    norm="l2"
).astype("float32")

print("Embedding shape:", knowledge_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shape: (137, 384)


In [26]:
np.save("../data/processed/knowledge_base2_embeddings.npy", knowledge_embeddings)

In [27]:
# create fiassindex

dimension = knowledge_embeddings.shape[1]

knowledge_index = faiss.IndexFlatIP(dimension)

knowledge_index.add(knowledge_embeddings)

print("Number of vectors:", knowledge_index.ntotal)
print("Vector dimension:", knowledge_index.d)

Number of vectors: 137
Vector dimension: 384


In [28]:
faiss.write_index(knowledge_index, "../data/processed/knowledge_base2_faiss.index")
print('saved')

saved


In [14]:
# test the business KB

In [29]:
knowledge_index = faiss.read_index("../data/processed/knowledge_base2_faiss.index")
knowledge_base_chunks = pd.read_csv("../data/processed/knowledge_base_chunks.csv")

print("FAISS vectors:", knowledge_index.ntotal)
print("Knowledge chunks:", len(knowledge_base_chunks))

FAISS vectors: 137
Knowledge chunks: 137


In [18]:
# create business knowledge search function

In [30]:
def search_business_knowledge(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = normalize(
        query_embedding,
        norm="l2"
    ).astype("float32")

    distances, indices = knowledge_index.search(
        query_embedding,
        top_k
    )

    results = knowledge_base_chunks.iloc[indices[0]].copy()
    results["similarity_score"] = distances[0]

    return results.reset_index(drop=True)

In [31]:
test_queries = [
    "How can I request a refund?",
    "My payment failed. What should I do?",
    "Can I cancel my order?",
    "My product stopped working after installation.",
    "My product is defective. Is it covered by warranty?",
    "My order says delivered but I did not receive it.",
    "I was charged twice for the same purchase.",
    "When should a customer issue be escalated?"
]

for query in test_queries:
    results = search_business_knowledge(query, top_k=3)

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for i, row in results.iterrows():
        print(
            f"{i + 1}. {row['document_id']} "
            f"| similarity: {row['similarity_score']:.4f}"
        )
        print(row["text"][:250])
        print()


QUERY: How can I request a refund?
1. refund_policy | similarity: 0.7101
If the available information is insufficient to determine refund eligibility, the request should be escalated to human customer support.

2. refund_policy | similarity: 0.6777
Refund Policy

3. refund_policy | similarity: 0.6393
Refund requests must be reviewed against the applicable purchase and refund conditions.


QUERY: My payment failed. What should I do?
1. payment_policy | similarity: 0.5574
If a payment problem cannot be verified from available information, the issue should be escalated to human customer support.

2. payment_policy | similarity: 0.5343
If a payment fails, the customer should verify the payment status before attempting another transaction.

3. billing_policy | similarity: 0.5336
If a payment was successful but the order was not confirmed, the payment status and order status should be checked before another payment is attempted.


QUERY: Can I cancel my order?
1. cancellation_policy | simil

In [21]:
results = search_business_knowledge(
    "My product stopped working after installation and factory reset did not fix it.",
    top_k=5
)

display(
    results[
        [
            "document_id",
            "source",
            "text",
            "similarity_score"
        ]
    ]
)

,document_id,source,text,similarity_score
0,technical_support,technical_support.txt,For a product that does not work after install...,0.542919
1,technical_support,technical_support.txt,If the customer has already performed a factor...,0.461303
2,technical_support,technical_support.txt,If a product stopped working after a software ...,0.431467
3,technical_support,technical_support.txt,Customers experiencing technical problems shou...,0.350195
4,warranty_policy,warranty_policy.txt,A product that stops working or arrives with a...,0.325722
